In [2]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.metrics import classification_report

In [9]:
import pandas as pd

train = pd.read_csv("train_processed.csv")

train.head()

,text,rating,tokens,stemmed,lemmatized,text_stem,text_lemma,review_length
0,this place is terrible the people in charge ar...,2,"['place', 'terrible', 'people', 'charge', 'wor...","['place', 'terribl', 'peopl', 'charg', 'worst'...","['place', 'terrible', 'people', 'charge', 'wor...",place terribl peopl charg worst part far yeah ...,place terrible people charge worst part far ye...,97
1,terrible service and they are saying that i ne...,1,"['terrible', 'service', 'saying', 'never', 'us...","['terribl', 'servic', 'say', 'never', 'use', '...","['terrible', 'service', 'saying', 'never', 'us...",terribl servic say never use servic lie call n...,terrible service saying never used service lie...,48
2,absolutely terrible company they sent me to co...,1,"['absolutely', 'terrible', 'company', 'sent', ...","['absolut', 'terribl', 'compani', 'sent', 'col...","['absolutely', 'terrible', 'company', 'sent', ...",absolut terribl compani sent collect without a...,absolutely terrible company sent collection wi...,211
3,to find it either park in front of the tuesday...,4,"['find', 'either', 'park', 'front', 'tuesday',...","['find', 'either', 'park', 'front', 'tuesday',...","['find', 'either', 'park', 'front', 'tuesday',...",find either park front tuesday morn mall entra...,find either park front tuesday morning mall en...,66
4,mall location used their services for sedan ni...,4,"['mall', 'location', 'used', 'services', 'seda...","['mall', 'locat', 'use', 'servic', 'sedan', 'n...","['mall', 'location', 'used', 'service', 'sedan...",mall locat use servic sedan nice perhap inform...,mall location used service sedan nice perhaps ...,28


In [6]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

In [10]:
train["text_lemma"] = train["text_lemma"].fillna("")
train["text_lemma"] = train["text_lemma"].astype(str)

train["text_stem"] = train["text_stem"].fillna("")
train["text_stem"] = train["text_stem"].astype(str)

train["text"] = train["text"].fillna("")
train["text"] = train["text"].astype(str)

X_d = train["text"]
X = train["text_lemma"]
x = train["text_stem"]
Y = train["rating"]

In [11]:
train.isnull().sum()

text             0
rating           0
tokens           0
stemmed          0
lemmatized       0
text_stem        0
text_lemma       0
review_length    0
dtype: int64

In [26]:
import pandas as pd
from sklearn.model_selection import GridSearchCV

def run_gridsearch(name, pipeline, params):
    # 1. Define multiple metrics
    scoring = {
        'Accuracy': 'accuracy',
        'MacroF1': 'f1_macro'
    }

    grid = GridSearchCV(
        pipeline,
        params,
        cv=3,
        n_jobs=-1,
        verbose=1,
        scoring=scoring,
        refit='MacroF1',
        return_train_score=False
    )

    grid.fit(X, Y)

    # 2. Extract detailed results into a table
    results_df = pd.DataFrame(grid.cv_results_)
    
    # Select columns for the 3 folds and the means
    columns_to_show = ['params']
    # Add Accuracy columns
    columns_to_show += ['split0_test_Accuracy', 'split1_test_Accuracy', 'split2_test_Accuracy', 'mean_test_Accuracy']
    # Add F1 columns
    columns_to_show += ['split0_test_MacroF1', 'split1_test_MacroF1', 'split2_test_MacroF1', 'mean_test_MacroF1']
    
    report_table = results_df[columns_to_show].sort_values('mean_test_MacroF1', ascending=False)

    print(f"\n==== {name} DETAILED FOLD RESULTS ====")
    print(report_table.to_string(index=False))

    # 3. Print the absolute bests
    best_f1_idx = results_df['mean_test_MacroF1'].idxmax()
    best_acc_idx = results_df['mean_test_Accuracy'].idxmax()

    print(f"\n--- {name} Summary ---")
    print(f"Best Params (for F1): {grid.best_params_}")
    print(f"Best Mean Macro-F1: {results_df.loc[best_f1_idx, 'mean_test_MacroF1']:.4f}")
    print(f"Best Mean Accuracy: {results_df.loc[best_acc_idx, 'mean_test_Accuracy']:.4f}")

    return grid

In [27]:
import joblib
from joblib import parallel_backend

nb_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", MultinomialNB())
])

nb_params = {
    "clf__alpha": [0.1, 0.5, 1.0],
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__max_features": [10000, 20000, 50000]
}

with parallel_backend('threading'):
    nb_grid = run_gridsearch(
        "Naive Bayes",
        nb_pipeline,
        nb_params
    )

Fitting 3 folds for each of 18 candidates, totalling 54 fits

==== Naive Bayes DETAILED FOLD RESULTS ====
                                                                         params  split0_test_Accuracy  split1_test_Accuracy  split2_test_Accuracy  mean_test_Accuracy  split0_test_MacroF1  split1_test_MacroF1  split2_test_MacroF1  mean_test_MacroF1
{'clf__alpha': 0.1, 'tfidf__max_features': 50000, 'tfidf__ngram_range': (1, 2)}              0.637219              0.636490              0.635531            0.636413             0.465647             0.465948             0.465692           0.465763
{'clf__alpha': 0.1, 'tfidf__max_features': 20000, 'tfidf__ngram_range': (1, 2)}              0.635490              0.634865              0.634031            0.634795             0.455449             0.456240             0.457453           0.456381
{'clf__alpha': 0.5, 'tfidf__max_features': 20000, 'tfidf__ngram_range': (1, 2)}              0.635573              0.634500              0.634583     

In [13]:
lr_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", LogisticRegression(max_iter=1000))
])

lr_params = {
    "clf__C": [0.5, 1, 3],
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__max_features": [10000, 20000, 50000]
}

with parallel_backend('threading'):
    lr_grid = run_gridsearch(
        "Logistic Regression",
        lr_pipeline,
        lr_params
    )

Fitting 3 folds for each of 18 candidates, totalling 54 fits

==== Logistic Regression DETAILED FOLD RESULTS ====
                                                                     params  split0_test_Accuracy  split1_test_Accuracy  split2_test_Accuracy  mean_test_Accuracy  split0_test_MacroF1  split1_test_MacroF1  split2_test_MacroF1  mean_test_MacroF1
  {'clf__C': 3, 'tfidf__max_features': 20000, 'tfidf__ngram_range': (1, 2)}              0.648646              0.646969              0.647500            0.647705             0.510599             0.509157             0.512691           0.510815
  {'clf__C': 3, 'tfidf__max_features': 10000, 'tfidf__ngram_range': (1, 2)}              0.652115              0.649823              0.650417            0.650785             0.510937             0.508166             0.511990           0.510364
  {'clf__C': 3, 'tfidf__max_features': 50000, 'tfidf__ngram_range': (1, 2)}              0.646417              0.645062              0.646969            0

In [14]:
svm_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
    )),
    ("clf", LinearSVC())
])

svm_params = {
    "clf__C": [0.5, 1, 2],
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__max_features": [10000, 20000, 50000]
}

with parallel_backend('threading'):
    svm_grid = run_gridsearch(
        "SVM",
        svm_pipeline,
        svm_params
    )

Fitting 3 folds for each of 18 candidates, totalling 54 fits

==== SVM DETAILED FOLD RESULTS ====
                                                                     params  split0_test_Accuracy  split1_test_Accuracy  split2_test_Accuracy  mean_test_Accuracy  split0_test_MacroF1  split1_test_MacroF1  split2_test_MacroF1  mean_test_MacroF1
  {'clf__C': 2, 'tfidf__max_features': 20000, 'tfidf__ngram_range': (1, 2)}              0.636656              0.635792              0.636021            0.636156             0.489737             0.489656             0.493750           0.491048
{'clf__C': 0.5, 'tfidf__max_features': 50000, 'tfidf__ngram_range': (1, 2)}              0.644813              0.643302              0.643271            0.643795             0.489780             0.489879             0.493298           0.490985
  {'clf__C': 1, 'tfidf__max_features': 50000, 'tfidf__ngram_range': (1, 2)}              0.632365              0.631375              0.631531            0.631757         

In [15]:
from sklearn.linear_model import SGDClassifier

sgd_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()), # Parameters handled by grid
    ("clf", SGDClassifier(max_iter=1000, tol=1e-3, random_state=25))
])


sgd_params = {
    "clf__alpha": [0.0001, 0.001, 0.01],
    "tfidf__ngram_range": [(1, 1), (1, 2)], 
    "tfidf__max_features": [10000,20000, 50000]
}

with parallel_backend('threading'):
    sgd_grid = run_gridsearch("SGD Classifier", sgd_pipeline, sgd_params)

Fitting 3 folds for each of 18 candidates, totalling 54 fits

==== SGD Classifier DETAILED FOLD RESULTS ====
                                                                            params  split0_test_Accuracy  split1_test_Accuracy  split2_test_Accuracy  mean_test_Accuracy  split0_test_MacroF1  split1_test_MacroF1  split2_test_MacroF1  mean_test_MacroF1
{'clf__alpha': 0.0001, 'tfidf__max_features': 10000, 'tfidf__ngram_range': (1, 2)}              0.635687              0.633729              0.633042            0.634153             0.403259             0.402127             0.403054           0.402813
{'clf__alpha': 0.0001, 'tfidf__max_features': 20000, 'tfidf__ngram_range': (1, 2)}              0.634667              0.632823              0.632656            0.633382             0.399555             0.399090             0.400321           0.399656
{'clf__alpha': 0.0001, 'tfidf__max_features': 10000, 'tfidf__ngram_range': (1, 1)}              0.633646              0.630354            

In [16]:
from sklearn.linear_model import RidgeClassifier

ridge_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()), 
    ("clf", RidgeClassifier(random_state=28))
])

ridge_params = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],  # Comparison requirement [cite: 39]
    "tfidf__max_features": [10000,20000, 50000],   # Representation requirement [cite: 38]
    "clf__alpha": [0.1, 1.0, 10.0]           # The "knob" for regularization
}

with parallel_backend('threading'):
    ridge_grid = run_gridsearch("Ridge Classifier", ridge_pipeline, ridge_params)

Fitting 3 folds for each of 18 candidates, totalling 54 fits

==== Ridge Classifier DETAILED FOLD RESULTS ====
                                                                          params  split0_test_Accuracy  split1_test_Accuracy  split2_test_Accuracy  mean_test_Accuracy  split0_test_MacroF1  split1_test_MacroF1  split2_test_MacroF1  mean_test_MacroF1
 {'clf__alpha': 1.0, 'tfidf__max_features': 50000, 'tfidf__ngram_range': (1, 2)}              0.642333              0.640552              0.640708            0.641198             0.476529             0.475884             0.479256           0.477223
 {'clf__alpha': 0.1, 'tfidf__max_features': 20000, 'tfidf__ngram_range': (1, 2)}              0.637708              0.635792              0.637271            0.636924             0.473907             0.472687             0.478451           0.475015
 {'clf__alpha': 1.0, 'tfidf__max_features': 20000, 'tfidf__ngram_range': (1, 2)}              0.645969              0.643896              0.64

In [17]:
import pandas as pd
from sklearn.model_selection import GridSearchCV

def run_gridsearch(name, pipeline, params):
    # 1. Define multiple metrics
    scoring = {
        'Accuracy': 'accuracy',
        'MacroF1': 'f1_macro'
    }

    grid_st = GridSearchCV(
        pipeline,
        params,
        cv=3,
        n_jobs=-1,
        verbose=1,
        scoring=scoring,
        refit='MacroF1',
        return_train_score=False
    )

    grid_st.fit(x, Y)

    # 2. Extract detailed results into a table
    results_df = pd.DataFrame(grid_st.cv_results_)
    
    # Select columns for the 3 folds and the means
    columns_to_show = ['params']
    # Add Accuracy columns
    columns_to_show += ['split0_test_Accuracy', 'split1_test_Accuracy', 'split2_test_Accuracy', 'mean_test_Accuracy']
    # Add F1 columns
    columns_to_show += ['split0_test_MacroF1', 'split1_test_MacroF1', 'split2_test_MacroF1', 'mean_test_MacroF1']
    
    report_table = results_df[columns_to_show].sort_values('mean_test_MacroF1', ascending=False)

    print(f"\n==== {name} DETAILED FOLD RESULTS ====")
    print(report_table.to_string(index=False))

    # 3. Print the absolute bests
    best_f1_idx = results_df['mean_test_MacroF1'].idxmax()
    best_acc_idx = results_df['mean_test_Accuracy'].idxmax()

    print(f"\n--- {name} Summary ---")
    print(f"Best Params (for F1): {grid_st.best_params_}")
    print(f"Best Mean Macro-F1: {results_df.loc[best_f1_idx, 'mean_test_MacroF1']:.4f}")
    print(f"Best Mean Accuracy: {results_df.loc[best_acc_idx, 'mean_test_Accuracy']:.4f}")

    return grid_st

In [18]:
import joblib
from joblib import parallel_backend

nb_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", MultinomialNB())
])

nb_params = {
    "clf__alpha": [0.1, 0.5, 1.0],
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__max_features": [10000, 20000, 50000]
}

with parallel_backend('threading'):
    nb_grid = run_gridsearch(
        "Naive Bayes",
        nb_pipeline,
        nb_params
    )

Fitting 3 folds for each of 18 candidates, totalling 54 fits

==== Naive Bayes DETAILED FOLD RESULTS ====
                                                                         params  split0_test_Accuracy  split1_test_Accuracy  split2_test_Accuracy  mean_test_Accuracy  split0_test_MacroF1  split1_test_MacroF1  split2_test_MacroF1  mean_test_MacroF1
{'clf__alpha': 0.1, 'tfidf__max_features': 50000, 'tfidf__ngram_range': (1, 2)}              0.635104              0.634896              0.633302            0.634434             0.462244             0.462969             0.462135           0.462449
{'clf__alpha': 0.1, 'tfidf__max_features': 20000, 'tfidf__ngram_range': (1, 2)}              0.633677              0.633135              0.631708            0.632840             0.452543             0.454890             0.453445           0.453626
{'clf__alpha': 0.5, 'tfidf__max_features': 20000, 'tfidf__ngram_range': (1, 2)}              0.634208              0.633521              0.631927     

In [19]:
lr_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", LogisticRegression(max_iter=1000))
])

lr_params = {
    "clf__C": [0.5, 1, 3],
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__max_features": [10000, 20000, 50000]
}

with parallel_backend('threading'):
    lr_grid = run_gridsearch(
        "Logistic Regression",
        lr_pipeline,
        lr_params
    )

Fitting 3 folds for each of 18 candidates, totalling 54 fits

==== Logistic Regression DETAILED FOLD RESULTS ====
                                                                     params  split0_test_Accuracy  split1_test_Accuracy  split2_test_Accuracy  mean_test_Accuracy  split0_test_MacroF1  split1_test_MacroF1  split2_test_MacroF1  mean_test_MacroF1
  {'clf__C': 3, 'tfidf__max_features': 20000, 'tfidf__ngram_range': (1, 2)}              0.648010              0.647042              0.646698            0.647250             0.509235             0.509063             0.512284           0.510194
  {'clf__C': 3, 'tfidf__max_features': 10000, 'tfidf__ngram_range': (1, 2)}              0.650927              0.650792              0.649615            0.650444             0.509764             0.508159             0.511294           0.509739
  {'clf__C': 3, 'tfidf__max_features': 50000, 'tfidf__ngram_range': (1, 2)}              0.644167              0.644635              0.643698            0

In [20]:
svm_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
    )),
    ("clf", LinearSVC())
])

svm_params = {
    "clf__C": [0.5, 1, 2],
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__max_features": [10000, 20000, 50000]
}

with parallel_backend('threading'):
    svm_grid = run_gridsearch(
        "SVM",
        svm_pipeline,
        svm_params
    )

Fitting 3 folds for each of 18 candidates, totalling 54 fits

==== SVM DETAILED FOLD RESULTS ====
                                                                     params  split0_test_Accuracy  split1_test_Accuracy  split2_test_Accuracy  mean_test_Accuracy  split0_test_MacroF1  split1_test_MacroF1  split2_test_MacroF1  mean_test_MacroF1
{'clf__C': 0.5, 'tfidf__max_features': 50000, 'tfidf__ngram_range': (1, 2)}              0.642458              0.641979              0.641698            0.642045             0.488456             0.488385             0.490384           0.489075
  {'clf__C': 1, 'tfidf__max_features': 20000, 'tfidf__ngram_range': (1, 2)}              0.641208              0.640437              0.640740            0.640795             0.487792             0.487575             0.491707           0.489024
  {'clf__C': 2, 'tfidf__max_features': 20000, 'tfidf__ngram_range': (1, 2)}              0.634854              0.633437              0.634781            0.634358         

In [21]:
from sklearn.linear_model import SGDClassifier

sgd_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()), # Parameters handled by grid
    ("clf", SGDClassifier(max_iter=1000, tol=1e-3, random_state=25))
])


sgd_params = {
    "clf__alpha": [0.0001, 0.001, 0.01],
    "tfidf__ngram_range": [(1, 1), (1, 2)], 
    "tfidf__max_features": [10000,20000, 50000]
}

with parallel_backend('threading'):
    sgd_grid = run_gridsearch("SGD Classifier", sgd_pipeline, sgd_params)

Fitting 3 folds for each of 18 candidates, totalling 54 fits

==== SGD Classifier DETAILED FOLD RESULTS ====
                                                                            params  split0_test_Accuracy  split1_test_Accuracy  split2_test_Accuracy  mean_test_Accuracy  split0_test_MacroF1  split1_test_MacroF1  split2_test_MacroF1  mean_test_MacroF1
{'clf__alpha': 0.0001, 'tfidf__max_features': 10000, 'tfidf__ngram_range': (1, 2)}              0.634062              0.633708              0.633646            0.633806             0.402792             0.402921             0.405125           0.403613
{'clf__alpha': 0.0001, 'tfidf__max_features': 20000, 'tfidf__ngram_range': (1, 2)}              0.633542              0.632656              0.632594            0.632931             0.399818             0.400024             0.401631           0.400491
{'clf__alpha': 0.0001, 'tfidf__max_features': 10000, 'tfidf__ngram_range': (1, 1)}              0.631469              0.629448            

In [22]:
from sklearn.linear_model import RidgeClassifier

ridge_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()), 
    ("clf", RidgeClassifier(random_state=28))
])

ridge_params = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],  # Comparison requirement [cite: 39]
    "tfidf__max_features": [10000,20000, 50000],   # Representation requirement [cite: 38]
    "clf__alpha": [0.1, 1.0, 10.0]           # The "knob" for regularization
}

with parallel_backend('threading'):
    ridge_grid = run_gridsearch("Ridge Classifier", ridge_pipeline, ridge_params)

Fitting 3 folds for each of 18 candidates, totalling 54 fits

==== Ridge Classifier DETAILED FOLD RESULTS ====
                                                                          params  split0_test_Accuracy  split1_test_Accuracy  split2_test_Accuracy  mean_test_Accuracy  split0_test_MacroF1  split1_test_MacroF1  split2_test_MacroF1  mean_test_MacroF1
 {'clf__alpha': 1.0, 'tfidf__max_features': 50000, 'tfidf__ngram_range': (1, 2)}              0.639760              0.638823              0.639448            0.639344             0.474423             0.473702             0.477432           0.475186
 {'clf__alpha': 0.1, 'tfidf__max_features': 20000, 'tfidf__ngram_range': (1, 2)}              0.637323              0.634927              0.636156            0.636135             0.473857             0.471603             0.477052           0.474171
 {'clf__alpha': 1.0, 'tfidf__max_features': 20000, 'tfidf__ngram_range': (1, 2)}              0.645250              0.643583              0.64

# Deep-Learning Model

In [23]:
train['rating'] = train['rating'] - 1

In [25]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Use the ORIGINAL review text here, not stemmed/lemmatized!
max_words = 10000
max_len = 100 # Max words per review

tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(train['text']) # Assuming 'text' is your raw column

sequences = tokenizer.texts_to_sequences(train['text'])
X_seq = pad_sequences(sequences, maxlen=max_len)
y_seq = train['rating'].values # Use the 0-4 labels you fixed earlier

In [ ]:
import pandas as pd
from sklearn.model_selection import GridSearchCV
import joblib
from joblib import parallel_backend

def run_gridsearch(name, pipeline, params):
    # 1. Define multiple metrics
    scoring = {
        'Accuracy': 'accuracy',
        'MacroF1': 'f1_macro'
    }

    grid = GridSearchCV(
        pipeline,
        params,
        cv=3,
        n_jobs=1,
        verbose=1,
        scoring=scoring,
        refit='MacroF1',
        return_train_score=False
    )

    # Use a context manager to ensure it doesn't fork
    with parallel_backend('threading'):
        grid.fit(X_seq, y_seq)

    # 2. Extract detailed results into a table
    results_df = pd.DataFrame(grid.cv_results_)

    # Select columns for the 3 folds and the means
    columns_to_show = ['params']
    # Add Accuracy columns
    columns_to_show += ['split0_test_Accuracy', 'split1_test_Accuracy', 'split2_test_Accuracy', 'mean_test_Accuracy']
    # Add F1 columns
    columns_to_show += ['split0_test_MacroF1', 'split1_test_MacroF1', 'split2_test_MacroF1', 'mean_test_MacroF1']

    report_table = results_df[columns_to_show].sort_values('mean_test_MacroF1', ascending=False)

    print(f"\n==== {name} DETAILED FOLD RESULTS ====")
    print(report_table.to_string(index=False))

    # 3. Print the absolute bests
    best_f1_idx = results_df['mean_test_MacroF1'].idxmax()
    best_acc_idx = results_df['mean_test_Accuracy'].idxmax()

    print(f"\n--- {name} Summary ---")
    print(f"Best Params (for F1): {grid.best_params_}")
    print(f"Best Mean Macro-F1: {results_df.loc[best_f1_idx, 'mean_test_MacroF1']:.4f}")
    print(f"Best Mean Accuracy: {results_df.loc[best_acc_idx, 'mean_test_Accuracy']:.4f}")

    return grid

In [ ]:
!pip uninstall -y scikit-learn
!pip install scikit-learn==1.5.2

Found existing installation: scikit-learn 1.5.2
Uninstalling scikit-learn-1.5.2:
  Successfully uninstalled scikit-learn-1.5.2
Defaulting to user installation because normal site-packages is not writeable
  Using cached scikit_learn-1.5.2-cp313-cp313-win_amd64.whl.metadata (13 kB)
Using cached scikit_learn-1.5.2-cp313-cp313-win_amd64.whl (11.0 MB)



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: C:\Users\hafee\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout, Input


# Force everything to run in the current thread
parallel_backend('threading')

# Pass max_len and max_words as defaults so the function is self-contained
def create_bilstm_model(lstm_units=64, dropout_rate=0.2, embedding_dim=100, max_len=100, max_words=10000):
    model = Sequential([
        Input(shape=(max_len,)), 

        Embedding(
            input_dim=max_words,
            output_dim=embedding_dim
        ),

        Bidirectional(LSTM(lstm_units)),
        Dropout(dropout_rate),
        Dense(32, activation="relu"),
        Dense(5, activation="softmax")
    ])

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

from scikeras.wrappers import KerasClassifier

# Wrap for Scikit-Learn
bilstm_clf = KerasClassifier(
    model=create_bilstm_model,
    model__max_len=max_len,     # Explicitly pass these
    model__max_words=max_words, # Explicitly pass these
    verbose=0
)

# Parameters to tune for your "Critical Analysis" [cite: 70]
bilstm_params = {
    "model__lstm_units": [32, 64, 128],
    "model__embedding_dim": [100, 200],
    "batch_size": [64],
    "epochs": [5]
}

bilstm_grid = run_gridsearch("BiLSTM", bilstm_clf, bilstm_params)

Fitting 3 folds for each of 6 candidates, totalling 18 fits

==== BiLSTM DETAILED FOLD RESULTS ====
                                                                                params  split0_test_Accuracy  split1_test_Accuracy  split2_test_Accuracy  mean_test_Accuracy  split0_test_MacroF1  split1_test_MacroF1  split2_test_MacroF1  mean_test_MacroF1
{'batch_size': 64, 'epochs': 5, 'model__embedding_dim': 200, 'model__lstm_units': 128}              0.654979              0.654677              0.651177            0.653611             0.528028             0.539777             0.539956           0.535920
 {'batch_size': 64, 'epochs': 5, 'model__embedding_dim': 100, 'model__lstm_units': 32}              0.657760              0.652635              0.647823            0.652740             0.529127             0.533680             0.542673           0.535160
 {'batch_size': 64, 'epochs': 5, 'model__embedding_dim': 200, 'model__lstm_units': 32}              0.653292              0.654385     

In [ ]:
from tensorflow.keras.layers import SpatialDropout1D, Bidirectional, LSTM, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout, Input, SpatialDropout1D, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.utils import class_weight
import numpy as np

def build_advanced_bilstm():
    model = Sequential([
        Input(shape=(max_len,)),
        # Increase embedding_dim to 300 if using GloVe 300d
        Embedding(input_dim=max_words, output_dim=100, mask_zero=True), 
        SpatialDropout1D(0.3),
        
        Bidirectional(LSTM(128, return_sequences=True)), # Stacked LSTM
        Bidirectional(LSTM(64)),
        
        BatchNormalization(), # Stabilizes learning
        Dropout(0.5),
        Dense(64, activation='relu'),
        Dense(5, activation='softmax')
    ])
    
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

# Add Callbacks to prevent the 58-minute timeout/overfit cycle
callbacks = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2)
]

# Train with Class Weights (Calculated from your y_seq)
from sklearn.utils import class_weight
import numpy as np

weights = class_weight.compute_class_weight('balanced', classes=np.unique(y_seq), y=y_seq)
class_weights_dict = dict(enumerate(weights))

model = build_advanced_bilstm()
history = model.fit(
    X_seq, y_seq,
    validation_split=0.2,
    epochs=15, # Callbacks will stop it early if it overfits
    batch_size=32, # Smaller batch size often helps accuracy
    class_weight=class_weights_dict,
    callbacks=callbacks
)

Epoch 1/15
7200/7200 ━━━━━━━━━━━━━━━━━━━━ 909s 126ms/step - accuracy: 0.5467 - loss: 1.0815 - val_accuracy: 0.5782 - val_loss: 0.9612 - learning_rate: 0.0010
Epoch 2/15
7200/7200 ━━━━━━━━━━━━━━━━━━━━ 976s 136ms/step - accuracy: 0.5949 - loss: 0.9766 - val_accuracy: 0.6030 - val_loss: 0.9317 - learning_rate: 0.0010
Epoch 3/15
7200/7200 ━━━━━━━━━━━━━━━━━━━━ 1061s 147ms/step - accuracy: 0.6139 - loss: 0.9294 - val_accuracy: 0.5886 - val_loss: 0.9498 - learning_rate: 0.0010
Epoch 4/15
7200/7200 ━━━━━━━━━━━━━━━━━━━━ 1212s 168ms/step - accuracy: 0.6309 - loss: 0.8938 - val_accuracy: 0.6245 - val_loss: 0.9112 - learning_rate: 0.0010
Epoch 5/15
7200/7200 ━━━━━━━━━━━━━━━━━━━━ 3725s 517ms/step - accuracy: 0.6452 - loss: 0.8623 - val_accuracy: 0.5877 - val_loss: 0.9621 - learning_rate: 0.0010
Epoch 6/15
7200/7200 ━━━━━━━━━━━━━━━━━━━━ 28646s 4s/step - accuracy: 0.6588 - loss: 0.8304 - val_accuracy: 0.5905 - val_loss: 0.9667 - learning_rate: 0.0010
Epoch 7/15
7200/7200 ━━━━━━━━━━━━━━━━━━━━ 907s 126

# BERT

In [29]:
# Install required libraries if not already present
%pip install transformers[torch] datasets accelerate scikit-learn

Defaulting to user installation because normal site-packages is not writeable
  Using cached datasets-4.8.4-py3-none-any.whl.metadata (19 kB)
  Using cached accelerate-1.13.0-py3-none-any.whl.metadata (19 kB)
  Using cached transformers-5.3.0-py3-none-any.whl.metadata (32 kB)
  Using cached huggingface_hub-1.7.2-py3-none-any.whl.metadata (13 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached typer-0.24.1-py3-none-any.whl.metadata (16 kB)
  Using cached torch-2.11.0-cp313-cp313-win_amd64.whl.metadata (29 kB)
  Using cached filelock-3.25.2-py3-none-any.whl.metadata (2.0 kB)
  Using cached fsspec-2026.2.0-py3-none-any.whl.metadata (10 kB)
  Using cached hf_xet-1.4.2-cp37-abi3-win_amd64.whl.metadata (4.9 kB)
  Using cached dill-0.4.1-py3-none-any.whl.metadata (10 kB)
  Using cached multiprocess-0.70.19-py313-none-any.whl.metadata (7.5 kB)
  Using cached aiohttp-3.13.3-cp313-cp313-win_amd64.whl.metadata (8.4 kB)
  Using cached aiohappyeyeballs-2.6.


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: C:\Users\hafee\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [30]:
import torch
import pandas as pd
import numpy as np
from datasets import Dataset
from transformers import (
    DistilBertTokenizerFast, 
    DistilBertForSequenceClassification, 
    Trainer, 
    TrainingArguments
)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score
from torch import nn

# Verify GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Working on: {device}")

Working on: cpu


In [31]:
# Load full dataset
train = pd.read_csv("train_processed.csv")
train['text'] = train['text'].fillna('').astype(str)

# Shift ratings to 0-4 for BERT
if train['rating'].max() == 5:
    train['rating'] = train['rating'] - 1

# Calculate balanced weights
weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train['rating']),
    y=train['rating']
)
class_weights = torch.tensor(weights, dtype=torch.float).to(device)

# Convert to Hugging Face Dataset
hf_dataset = Dataset.from_pandas(train[['text', 'rating']].rename(columns={'rating': 'labels'}))
dataset = hf_dataset.train_test_split(test_size=0.1)

# Tokenization
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=128)

tokenized_dataset = dataset.map(tokenize, batched=True)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/259200 [00:00<?, ? examples/s]

Map:   0%|          | 0/28800 [00:00<?, ? examples/s]

In [33]:
class WeightedTrainer(Trainer):
    # We add 'num_items_in_batch=None' and '**kwargs' to match the new library requirements
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None, **kwargs):
        labels = inputs.get("labels")
        
        # Standard Forward pass
        outputs = model(**inputs)
        logits = outputs.get("logits")
        
        # Our custom Weighted Loss logic (stays the same)
        loss_fct = nn.CrossEntropyLoss(weight=class_weights)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions, average="macro")
    }

In [34]:
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", 
    num_labels=5
).to(device)

training_args = TrainingArguments(
    output_dir="./distilbert_final",
    num_train_epochs=2,              # 2 epochs is the sweet spot for 288k rows
    per_device_train_batch_size=32,  # Optimized for 8GB+ VRAM GPUs
    eval_strategy="epoch",           # Evaluate once per hour to save time
    save_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    fp16=True,                       # Enables Mixed Precision (Speed Boost)
    load_best_model_at_end=True,
    report_to="none"
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    compute_metrics=compute_metrics
)

# This will likely take 1.5 to 2.5 hours total
trainer.train()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
C:\Users\hafee\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no ac

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [35]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"Is CUDA available? {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: CUDA is NOT detected. The code is using the slow CPU.")

PyTorch version: 2.11.0+cpu
Is CUDA available? False


In [ ]:
pip uninstall torch torchvision torchaudio -y
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121